# Variant B — Explanation-Augmented Input

This notebook trains and evaluates Variant B of the DeBERTa-v3-based NLI model.

**Architecture:** `[CLS] premise hypothesis [SEP] explanation [SEP]` → classification

**Test-time explanation strategies:**
1. Gold explanations (e-SNLI test only — upper bound)
2. TF-IDF retrieval from training set
3. T5-small generated explanations

**Estimated total time on T4:** ~2.5–3 hours

## 1. Setup & Dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn joblib

In [ ]:
import sys
import os
import torch

# Mount Google Drive for checkpointing
from google.colab import drive
drive.mount('/content/drive')

# Clone or set up project path
# Update this path to match your Google Drive project location
PROJECT_ROOT = "/content/drive/MyDrive/project"
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
from training.config import VariantBConfig

config = VariantBConfig()
print(f"Model: {config.model_name}")
print(f"Max seq length: {config.max_seq_length}")
print(f"Batch size: {config.per_device_train_batch_size} x {config.gradient_accumulation_steps} = {config.per_device_train_batch_size * config.gradient_accumulation_steps}")
print(f"Epochs: {config.num_train_epochs}")
print(f"Output dir: {config.output_dir}")

## 2. Fine-tune T5-small for Explanation Generation (~25 min)

Train T5-small first so it's ready when we need it for evaluation later.

In [ ]:
from data.t5_explanation import finetune_t5

t5_model, t5_tokenizer = finetune_t5(config)
print("T5-small fine-tuning complete!")

In [ ]:
# Quick sanity check: generate a few explanations
from data.t5_explanation import generate_explanations

sample_premises = [
    "A man is playing guitar on stage.",
    "Two dogs are running in the park.",
]
sample_hypotheses = [
    "A man is performing music.",
    "The animals are sleeping.",
]

sample_explanations = generate_explanations(
    t5_model, t5_tokenizer, sample_premises, sample_hypotheses,
    batch_size=2, device="cuda"
)
for p, h, e in zip(sample_premises, sample_hypotheses, sample_explanations):
    print(f"P: {p}")
    print(f"H: {h}")
    print(f"E: {e}")
    print()

In [ ]:
# Free T5 GPU memory before DeBERTa training
del t5_model
torch.cuda.empty_cache()
print("T5 model removed from GPU memory.")

## 3. Train DeBERTa Variant B (~2 hours)

Main training: DeBERTa-v3-base with premise + hypothesis + explanation as input.

In [ ]:
import os
import torch

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

from training.train_variant_b import train

# To resume from a checkpoint after a Colab disconnect, uncomment:
# trainer, model = train(config, resume_from_checkpoint="results/variant_b/checkpoints/checkpoint-XXXX")

trainer, model = train(config)
print("DeBERTa Variant B training complete!")

## 4. Build TF-IDF Index (~10 min)

In [ ]:
from data.tfidf_retrieval import build_tfidf_index, save_tfidf_index

vectorizer, train_matrix, train_explanations = build_tfidf_index(config)

tfidf_dir = str(config.get_path("output_dir")) + "/tfidf_index"
save_tfidf_index(vectorizer, train_matrix, train_explanations, tfidf_dir)
print("TF-IDF index built and saved.")

## 5. Full Evaluation — All Strategies, All Benchmarks (~20 min)

In [ ]:
from evaluation.evaluate_variant_b import run_full_evaluation
import json

model_dir = str(config.get_path("output_dir"))
t5_model_dir = str(config.get_path("t5_output_dir"))

results = run_full_evaluation(
    model_dir=model_dir,
    config=config,
    device="cuda",
    tfidf_dir=tfidf_dir,
    t5_model_dir=t5_model_dir,
)

print("\n" + "=" * 60)
print("VARIANT B — FULL RESULTS")
print("=" * 60)
print(json.dumps(results, indent=2))

In [ ]:
# Save results to JSON
from pathlib import Path

output_path = Path(config.get_path("output_dir")) / "eval_results.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {output_path}")